In [2]:
import os
import csv
import random
from glob import glob
from typing import Tuple, Dict

import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext

try:
    from tqdm import tqdm
except Exception:
    tqdm = None


# ============================================================
# PATH CONFIG — EDIT THESE FOR YOUR LOCAL MACHINE
# ============================================================

IMG_TRAIN = r"G:\historical document analysis\Latin2\images\train"
IMG_VAL = r"G:\historical document analysis\Latin2\images\val"
IMG_TEST = r"G:\historical document analysis\Latin2\images\test"

TL_MSK_TRAIN = r"G:\historical document analysis\Latin2\masks\train"
TL_MSK_VAL = r"G:\historical document analysis\Latin2\masks\val"
TL_MSK_TEST = r"G:\historical document analysis\Latin2\masks\test"

PRIOR_DIR_TRAIN = r"G:\historical document analysis\Latin2\priors\train"
PRIOR_DIR_VAL = r"G:\historical document analysis\Latin2\priors\val"
PRIOR_DIR_TEST = r"G:\historical document analysis\Latin2\priors\test"

# We will generate priors dynamically using the Stage 1 model.
STAGE1_CKPT = r"G:\Best UDIDS\best_maskprob_gauss.pt"

OUT_DIR = r"sensitivity_output"


# ============================================================
# TRAINING CONFIG
# ============================================================

EPOCHS = 250  # Changed from 500 to 50 for quick single-image testing
BATCH_SIZE = 1
BASE = 48
LR_INIT = 1e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP = 1.0

MAX_SIDE = 2048
PATCH = 512
TRAIN_PATCHES_PER_IMAGE = 8
POS_PATCH_PROB = 0.6
VAL_STRIDE = PATCH // 2

NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

CASCADE_USE_PRIOR_AS_INPUT = True

SEED = 42
ALLOWED = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


# ============================================================
# SENSITIVITY SETTINGS
# ============================================================

RUNS = [
    {
        "name": "beta_0.0",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 0.0,
    },
    {
        "name": "beta_0.5",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 0.5,
    },
    {
        "name": "beta_1.0",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 1.0,
    },
    {
        "name": "beta_1.5_baseline",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 1.5,
    },
    {
        "name": "beta_2.0",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 2.0,
    },
    {
        "name": "beta_3.0",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 3.0,
    },
    {
        "name": "beta_5.0",
        "alpha_min": 0.15,
        "alpha_max": 1.50,
        "beta_gate": 5.0,
    },
]


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# I/O
# ============================================================

def load_rgb(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


def load_mask01(path: str) -> np.ndarray:
    return (np.array(Image.open(path).convert("L")) > 0).astype(np.uint8)


def resize_keep(img: np.ndarray, max_side: int = MAX_SIDE):
    h, w = img.shape[:2]
    s = max(h, w)

    if s <= max_side:
        return img, 1.0

    scale = max_side / float(s)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    if img.ndim == 3:
        interpolation = cv2.INTER_AREA
    else:
        interpolation = cv2.INTER_NEAREST

    out = cv2.resize(img, (new_w, new_h), interpolation=interpolation)
    return out, scale


# ============================================================
# DATASET
# ============================================================

class TextLineDataset(Dataset):
    def __init__(self, img_dir: str, mask_dir: str, prior_dir: str, stage1_model: nn.Module = None, device: torch.device = None, use_prior: bool = True):
        if not os.path.isdir(img_dir):
            raise FileNotFoundError(f"Image directory not found: {img_dir}")
        if not os.path.isdir(mask_dir):
            raise FileNotFoundError(f"Mask directory not found: {mask_dir}")

        img_paths = sorted(
            p for p in glob(os.path.join(img_dir, "*"))
            if os.path.splitext(p)[1].lower() in ALLOWED
        )

        mask_paths = sorted(
            p for p in glob(os.path.join(mask_dir, "*"))
            if os.path.splitext(p)[1].lower() in ALLOWED
        )

        mask_map = {
            os.path.splitext(os.path.basename(p))[0]: p
            for p in mask_paths
        }

        self.items = []
        for ip in img_paths:
            key = os.path.splitext(os.path.basename(ip))[0]
            if key in mask_map:
                self.items.append((ip, mask_map[key], key))
            else:
                print(f"[warning] no mask found for image: {ip}")

        if not self.items:
            raise RuntimeError(f"No image/mask pairs found between {img_dir} and {mask_dir}")

        self.prior_dir = prior_dir
        self.stage1_model = stage1_model
        self.device = device
        self.use_prior = use_prior

        print(f"[dataset] {len(self.items)} pairs loaded from {img_dir}")

    def __len__(self):
        return len(self.items)

    def _resize_keep_aspect_img(self, img_np: np.ndarray, max_side: int = 768):
        H, W = img_np.shape
        s = max(H, W)
        if s <= max_side:
            return img_np, 1.0
        scale = max_side / float(s)
        newW = int(round(W * scale))
        newH = int(round(H * scale))
        img_res = np.array(Image.fromarray(img_np).resize((newW, newH), Image.BILINEAR))
        return img_res, scale

    def _get_or_create_prior(self, key: str, img_path: str):
        npy_path = os.path.join(self.prior_dir, f"{key}.npy")
        png_path = os.path.join(self.prior_dir, f"{key}.png")

        if os.path.isfile(npy_path):
            prior = np.load(npy_path).astype(np.float32)
            if prior.max() > 1.0:
                prior = prior / 255.0
            return prior
        elif os.path.isfile(png_path):
            prior = np.array(Image.open(png_path).convert("L")).astype(np.float32) / 255.0
            return prior

        # Compute it using Stage 1 model
        if self.stage1_model is None:
            print(f"[warning] Missing prior for '{key}' and no stage1_model provided. Using empty.")
            return None

        img_rgb = load_rgb(img_path)
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        gray_res, _ = self._resize_keep_aspect_img(gray, max_side=768)
        
        x_t = torch.from_numpy(gray_res.astype(np.float32) / 255.0)[None, None, ...].to(self.device)
        
        self.stage1_model.eval()
        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=(self.device.type == "cuda")):
                p = self.stage1_model(x_t)
            
        prior_res = p[0, 0].cpu().to(torch.float32).numpy()
        
        prior_full = cv2.resize(prior_res, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_LINEAR)
        prior_full = np.clip(prior_full, 0.0, 1.0).astype(np.float32)
        
        os.makedirs(self.prior_dir, exist_ok=True)
        np.save(npy_path, prior_full)
        return prior_full

    def _load_prior(self, key: str, img_path: str, h0: int, w0: int) -> np.ndarray:
        if not self.use_prior:
            return np.zeros((h0, w0), np.float32)

        prior_res = self._get_or_create_prior(key, img_path)
        if prior_res is None:
            return np.zeros((h0, w0), np.float32)

        if prior_res.shape != (h0, w0):
            prior = cv2.resize(prior_res, (w0, h0), interpolation=cv2.INTER_LINEAR)
        else:
            prior = prior_res
            
        return np.clip(prior, 0.0, 1.0).astype(np.float32)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        img_path, mask_path, key = self.items[idx]

        img0 = load_rgb(img_path)
        mask0 = load_mask01(mask_path)

        h0, w0 = mask0.shape
        prior0 = self._load_prior(key, img_path, h0, w0)

        img, _ = resize_keep(img0, MAX_SIDE)
        h, w = img.shape[:2]

        mask = cv2.resize(mask0, (w, h), interpolation=cv2.INTER_NEAREST).astype(np.float32)
        prior = cv2.resize(prior0, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        prior = np.clip(prior, 0.0, 1.0)

        rgb = (img.astype(np.float32) / 255.0).transpose(2, 0, 1)

        if self.use_prior:
            x = np.concatenate([rgb, prior[None, ...]], axis=0)
        else:
            x = rgb

        return {
            "x": torch.from_numpy(x).float(),
            "y": torch.from_numpy(mask[None, ...]).float(),
            "prior": torch.from_numpy(prior[None, ...]).float(),
            "rgb": torch.from_numpy(rgb).float(),
            "name": os.path.basename(img_path),
        }


# ============================================================
# STAGE 1 MODEL (For Priors)
# ============================================================

def _gn_groups(C: int) -> int:
    for g in (32, 16, 8, 4, 2, 1):
        if C % g == 0:
            return g
    return 1

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        g = _gn_groups(out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x): return self.net(x)

class DownOld(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x): return self.conv(self.pool(x))

class UpOld(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class UNetMaskProb(nn.Module):
    def __init__(self, in_ch=1, base=32):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = DownOld(base, base*2)
        self.e3 = DownOld(base*2, base*4)
        self.e4 = DownOld(base*4, base*8)
        self.bott = DoubleConv(base*8, base*16)
        self.u4 = UpOld(base*16, base*8, base*8)
        self.u3 = UpOld(base*8,  base*4, base*4)
        self.u2 = UpOld(base*4,  base*2, base*2)
        self.u1 = UpOld(base*2,  base,   base)
        self.head = nn.Conv2d(base, 1, 1)   # single mask-prob head

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        z = self.bott(s4)
        z = self.u4(z, s4)
        z = self.u3(z, s3)
        z = self.u2(z, s2)
        z = self.u1(z, s1)
        return torch.sigmoid(self.head(z))  # [B,1,H,W] in [0,1]


# ============================================================
# STAGE 2 MODEL
# ============================================================

def group_norm_groups(channels: int) -> int:
    for g in (32, 16, 8, 4, 2, 1):
        if channels % g == 0:
            return g
    return 1


class Block(nn.Module):
    def __init__(self, c_in: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        g = group_norm_groups(c_out)

        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.GroupNorm(g, c_out),
            nn.ReLU(inplace=True),

            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.GroupNorm(g, c_out),
            nn.ReLU(inplace=True),

            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.net(x)


class Down(nn.Module):
    def __init__(self, c_in: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.block = Block(c_in, c_out, dropout)

    def forward(self, x):
        return self.block(self.pool(x))


class Up(nn.Module):
    def __init__(self, c_in: int, c_skip: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        self.reduce = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.block = Block(c_out + c_skip, c_out, dropout)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)


def safe_logit(p: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    p = p.clamp(eps, 1.0 - eps)
    return torch.log(p) - torch.log(1.0 - p)


class UNetTextLines(nn.Module):
    def __init__(
        self,
        in_ch: int = 4,
        base: int = 64,
        dropout: float = 0.1,
        alpha_min: float = 0.15,
        alpha_max: float = 1.50,
        beta_gate: float = 1.5,
        res_scale: float = 2.0,
    ):
        super().__init__()

        self.in_ch = in_ch
        self.alpha_min = float(alpha_min)
        self.alpha_max = float(alpha_max)
        self.beta_gate = float(beta_gate)
        self.res_scale = float(res_scale)

        self.e1 = Block(in_ch, base, dropout)
        self.e2 = Down(base, base * 2, dropout)
        self.e3 = Down(base * 2, base * 4, dropout)
        self.e4 = Down(base * 4, base * 8, dropout)

        self.bott = Block(base * 8, base * 16, dropout)

        self.u4 = Up(base * 16, base * 8, base * 8, dropout)
        self.u3 = Up(base * 8, base * 4, base * 4, dropout)
        self.u2 = Up(base * 4, base * 2, base * 2, dropout)
        self.u1 = Up(base * 2, base, base, 0.0)

        self.head_delta = nn.Conv2d(base, 2, 1)
        self.head_gate = nn.Conv2d(base, 1, 1)
        self.head_alpha = nn.Conv2d(base, 1, 1)

        nn.init.zeros_(self.head_delta.weight)
        nn.init.zeros_(self.head_delta.bias)

        nn.init.zeros_(self.head_gate.weight)
        nn.init.constant_(self.head_gate.bias, -2.0)

        nn.init.zeros_(self.head_alpha.weight)
        nn.init.constant_(self.head_alpha.bias, 0.0)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)

        z = self.bott(s4)

        z = self.u4(z, s4)
        z = self.u3(z, s3)
        z = self.u2(z, s2)
        z = self.u1(z, s1)

        if self.in_ch >= 4:
            prior = x[:, 3:4].clamp(0.0, 1.0)

            l_fg = safe_logit(prior)
            l_bg = safe_logit(1.0 - prior)

            prior_logits = torch.cat([l_bg, l_fg], dim=1)
            prior_conf_mask = prior.pow(self.beta_gate)
        else:
            b, _, h, w = x.shape
            prior_logits = torch.zeros(b, 2, h, w, device=x.device, dtype=x.dtype)
            prior_conf_mask = torch.zeros(b, 1, h, w, device=x.device, dtype=x.dtype)

        delta = self.res_scale * torch.tanh(self.head_delta(z))
        delta = prior_conf_mask * delta

        gate = torch.sigmoid(self.head_gate(z))

        alpha_raw = torch.sigmoid(self.head_alpha(z))
        alpha_map = self.alpha_min + (self.alpha_max - self.alpha_min) * alpha_raw

        fused_logits = alpha_map * prior_logits + gate * delta

        return fused_logits, alpha_map


# ============================================================
# METRICS
# ============================================================

def evaluate_metrics_np(gt_u8: np.ndarray, pr_u8: np.ndarray, thresh: float = 0.75):
    ng, gt_lbl = cv2.connectedComponents(gt_u8)
    npred, pr_lbl = cv2.connectedComponents(pr_u8)

    inter = np.logical_and(gt_lbl > 0, pr_lbl > 0).sum()
    union = np.logical_or(gt_lbl > 0, pr_lbl > 0).sum()
    pixel_iou = inter / union if union > 0 else 0.0

    m = ng
    n = npred

    if m <= 1 or n <= 1:
        return pixel_iou, 0.0, 0.0, 0.0, 0.0

    table = np.bincount(
        gt_lbl.ravel() * n + pr_lbl.ravel(),
        minlength=m * n
    ).reshape(m, n)

    area_gt = table.sum(axis=1)[:, None]
    area_pr = table.sum(axis=0)[None, :]

    iou = table / (area_gt + area_pr - table + 1e-8)

    rows, cols = np.where(
        (iou >= thresh)
        & (np.arange(m)[:, None] > 0)
        & (np.arange(n)[None, :] > 0)
    )

    matches = len(rows)

    dr = matches / max(1, (m - 1))
    ra = matches / max(1, (n - 1))
    fm = 2 * dr * ra / (dr + ra + 1e-8)

    # Same Line IU implementation as your code
    sub = iou[1:, 1:]
    best_pr = sub.argmax(axis=1) + 1
    gt_i = np.arange(1, m)
    pr_i = best_pr

    ints = table[gt_i, pr_i]
    precs = ints / (area_pr[0, pr_i] + 1e-8)
    recs = ints / (area_gt[gt_i, 0] + 1e-8)

    cl = (precs >= thresh) & (recs >= thresh)
    ml = recs < thresh
    el = (precs < thresh) & (recs >= thresh)

    line_iu = cl.sum() / max(1, (cl.sum() + ml.sum() + el.sum()))

    return pixel_iou, line_iu, dr, ra, fm


def cosine_window_2d(h: int, w: int, eps: float = 1e-3) -> np.ndarray:
    wy = np.hanning(h) if h > 1 else np.ones(1)
    wx = np.hanning(w) if w > 1 else np.ones(1)
    win = np.outer(wy, wx).astype(np.float32)
    return np.clip(win, eps, 1.0)


def sample_random_patch(h: int, w: int, ph: int, pw: int, yc=None, xc=None):
    if yc is not None and xc is not None:
        y0 = max(0, min(h - ph, int(yc - ph // 2)))
        x0 = max(0, min(w - pw, int(xc - pw // 2)))
        return y0, x0

    y0 = 0 if h <= ph else random.randint(0, h - ph)
    x0 = 0 if w <= pw else random.randint(0, w - pw)
    return y0, x0


# ============================================================
# DATALOADER
# ============================================================

def make_loader(ds: Dataset, shuffle: bool):
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


# ============================================================
# TRAIN ONE SENSITIVITY RUN
# ============================================================

def train_one_run(run_cfg: dict):
    run_name = run_cfg["name"]
    alpha_min = run_cfg["alpha_min"]
    alpha_max = run_cfg["alpha_max"]
    beta_gate = run_cfg["beta_gate"]

    set_seed(SEED)

    run_dir = os.path.join(OUT_DIR, run_name)
    preview_dir = os.path.join(run_dir, "preview")

    os.makedirs(run_dir, exist_ok=True)
    os.makedirs(preview_dir, exist_ok=True)

    ckpt_best = os.path.join(run_dir, "best_lineiu.pt")
    ckpt_last = os.path.join(run_dir, "last_lineiu.pt")
    log_csv = os.path.join(run_dir, "metrics_lineiu.csv")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nRun: {run_name}")
    print(f"Device: {device}")
    print(f"alpha_min={alpha_min}, alpha_max={alpha_max}, beta_gate={beta_gate}")

    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass

    # Load Stage 1 model for prior generation
    stage1_model = None
    if CASCADE_USE_PRIOR_AS_INPUT:
        stage1_model = UNetMaskProb(in_ch=1, base=32).to(device)
        state = torch.load(STAGE1_CKPT, map_location=device)
        stage1_model.load_state_dict(state["model"], strict=True)
        stage1_model.eval()
        print(f"Loaded Stage 1 model from {STAGE1_CKPT} for dynamic prior generation")

    ds_train = TextLineDataset(
        IMG_TRAIN,
        TL_MSK_TRAIN,
        prior_dir=PRIOR_DIR_TRAIN,
        stage1_model=stage1_model,
        device=device,
        use_prior=CASCADE_USE_PRIOR_AS_INPUT,
    )

    ds_val = TextLineDataset(
        IMG_VAL,
        TL_MSK_VAL,
        prior_dir=PRIOR_DIR_VAL,
        stage1_model=stage1_model,
        device=device,
        use_prior=CASCADE_USE_PRIOR_AS_INPUT,
    )

    ds_test = TextLineDataset(
        IMG_TEST,
        TL_MSK_TEST,
        prior_dir=PRIOR_DIR_TEST,
        stage1_model=stage1_model,
        device=device,
        use_prior=CASCADE_USE_PRIOR_AS_INPUT,
    )

    train_loader = make_loader(ds_train, shuffle=True)
    val_loader = make_loader(ds_val, shuffle=False)
    test_loader = make_loader(ds_test, shuffle=False)

    in_ch = 4 if CASCADE_USE_PRIOR_AS_INPUT else 3

    model = UNetTextLines(
        in_ch=in_ch,
        base=BASE,
        dropout=0.1,
        alpha_min=alpha_min,
        alpha_max=alpha_max,
        beta_gate=beta_gate,
    ).to(device)

    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR_INIT,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=LR_INIT * 0.1,
    )

    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    ce_loss = nn.CrossEntropyLoss()

    best_lineiu = -1.0
    best_row = None

    for epoch in range(1, EPOCHS + 1):
        # -------------------------
        # Train
        # -------------------------
        model.train()
        train_loss = 0.0

        train_iter = train_loader

        for batch in train_iter:
            x_full = batch["x"].to(device, non_blocking=True).float()
            y_full = batch["y"].to(device, non_blocking=True).float()

            if device.type == "cuda":
                x_full = x_full.to(memory_format=torch.channels_last)
                y_full = y_full.to(memory_format=torch.channels_last)

            _, _, h, w = x_full.shape

            for _ in range(TRAIN_PATCHES_PER_IMAGE):
                yc = None
                xc = None

                if random.random() < POS_PATCH_PROB:
                    pos = torch.nonzero(y_full[0, 0] > 0.5, as_tuple=False)
                    if pos.numel() > 0:
                        selected = pos[random.randrange(pos.shape[0])].tolist()
                        yc, xc = selected[0], selected[1]

                y0, x0 = sample_random_patch(h, w, PATCH, PATCH, yc, xc)
                y1 = min(h, y0 + PATCH)
                x1 = min(w, x0 + PATCH)

                x = x_full[:, :, y0:y1, x0:x1]
                y = y_full[:, :, y0:y1, x0:x1]

                target = (y > 0.5).long().squeeze(1)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast('cuda', enabled=use_amp):
                    fused_logits, _ = model(x)
                    loss = ce_loss(fused_logits, target)

                scaler.scale(loss).backward()

                if GRAD_CLIP is not None and GRAD_CLIP > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

                scaler.step(optimizer)
                scaler.update()

                train_loss += float(loss.item()) * x.size(0)

        train_loss /= max(1, len(ds_train) * TRAIN_PATCHES_PER_IMAGE)

        # -------------------------
        # Validation
        # -------------------------
        model.eval()

        val_ce = 0.0
        val_dice_list = []
        val_pixel_iou_list = []
        val_lineiu_list = []
        val_dr_list = []
        val_ra_list = []
        val_fm_list = []
        alpha_means = []

        val_iter = val_loader

        with torch.no_grad():
            for i, batch in enumerate(val_iter):
                x_full = batch["x"][0].to(device).float()
                y_full = batch["y"][0].to(device).float()

                c, h, w = x_full.shape

                acc = torch.zeros((1, h, w), device=device, dtype=torch.float32)
                wgt = torch.zeros((1, h, w), device=device, dtype=torch.float32)

                acc_alpha = torch.zeros((1, h, w), device=device, dtype=torch.float32)
                wgt_alpha = torch.zeros((1, h, w), device=device, dtype=torch.float32)

                ys = list(range(0, max(1, h - PATCH + 1), VAL_STRIDE))
                xs = list(range(0, max(1, w - PATCH + 1), VAL_STRIDE))

                edge_y = max(0, h - PATCH)
                edge_x = max(0, w - PATCH)

                if ys[-1] != edge_y:
                    ys.append(edge_y)
                if xs[-1] != edge_x:
                    xs.append(edge_x)

                win_cache = {}

                for y0 in ys:
                    for x0 in xs:
                        y1 = min(h, y0 + PATCH)
                        x1 = min(w, x0 + PATCH)

                        tile = x_full[:, y0:y1, x0:x1].unsqueeze(0).to(device)

                        fused_logits, alpha_map = model(tile)
                        prob = torch.softmax(fused_logits, dim=1)[:, 1:2]

                        th = y1 - y0
                        tw = x1 - x0

                        key = (th, tw)
                        if key not in win_cache:
                            win_cache[key] = torch.from_numpy(
                                cosine_window_2d(th, tw)
                            ).to(device=device, dtype=torch.float32)

                        win = win_cache[key]

                        acc[:, y0:y1, x0:x1] += prob.squeeze(0) * win
                        wgt[:, y0:y1, x0:x1] += win

                        acc_alpha[:, y0:y1, x0:x1] += alpha_map.squeeze(0) * win
                        wgt_alpha[:, y0:y1, x0:x1] += win

                prob_full = torch.where(wgt > 0, acc / wgt, acc).clamp(0.0, 1.0)
                alpha_full = torch.where(wgt_alpha > 0, acc_alpha / wgt_alpha, acc_alpha)
                alpha_means.append(float(alpha_full.mean().cpu()))

                # Validation CE reconstructed from probability
                p = prob_full.clamp(1e-6, 1.0 - 1e-6)
                l_fg = torch.log(p) - torch.log(1.0 - p)
                l_bg = -l_fg
                logits_full = torch.cat([l_bg, l_fg], dim=0).unsqueeze(0)

                target_full = (y_full > 0.5).long().squeeze(0).unsqueeze(0)
                loss_val = ce_loss(logits_full, target_full)
                val_ce += float(loss_val.item())

                pred = (prob_full >= 0.5).float()
                gt = (y_full > 0.5).float()

                tp = (pred * gt).sum().item()
                fp = (pred * (1.0 - gt)).sum().item()
                fn = ((1.0 - pred) * gt).sum().item()

                dice = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-6)
                val_dice_list.append(dice)

                gt_u8 = (gt[0].cpu().numpy().astype(np.uint8)) * 255
                pr_u8 = (pred[0].cpu().numpy().astype(np.uint8)) * 255

                pixel_iou, line_iu, dr, ra, fm = evaluate_metrics_np(gt_u8, pr_u8, thresh=0.75)

                val_pixel_iou_list.append(pixel_iou)
                val_lineiu_list.append(line_iu)
                val_dr_list.append(dr)
                val_ra_list.append(ra)
                val_fm_list.append(fm)

                # Save preview only occasionally
                if i == 0 and (epoch == 1 or epoch % 25 == 0):
                    rgb = (batch["rgb"][0].numpy().transpose(1, 2, 0) * 255).astype(np.uint8)
                    rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

                    pred_u8 = (prob_full[0].cpu().numpy() * 255).astype(np.uint8)
                    gt_u8_vis = (gt[0].cpu().numpy() * 255).astype(np.uint8)

                    pred_color = cv2.applyColorMap(pred_u8, cv2.COLORMAP_JET)
                    gt_color = cv2.applyColorMap(gt_u8_vis, cv2.COLORMAP_JET)

                    ov_pred = cv2.addWeighted(rgb_bgr, 0.5, pred_color, 0.6, 0)
                    ov_gt = cv2.addWeighted(rgb_bgr, 0.5, gt_color, 0.6, 0)

                    cv2.imwrite(os.path.join(preview_dir, f"ep{epoch:03d}_img.png"), rgb_bgr)
                    cv2.imwrite(os.path.join(preview_dir, f"ep{epoch:03d}_pred_prob.png"), pred_u8)
                    cv2.imwrite(os.path.join(preview_dir, f"ep{epoch:03d}_gt.png"), gt_u8_vis)
                    cv2.imwrite(os.path.join(preview_dir, f"ep{epoch:03d}_ov_pred.png"), ov_pred)
                    cv2.imwrite(os.path.join(preview_dir, f"ep{epoch:03d}_ov_gt.png"), ov_gt)

                    prior_u8 = (batch["prior"][0, 0].numpy().clip(0, 1) * 255).astype(np.uint8)
                    cv2.imwrite(os.path.join(preview_dir, f"ep{epoch:03d}_prior.png"), prior_u8)

        val_ce /= max(1, len(ds_val))
        val_dice = float(np.mean(val_dice_list)) if val_dice_list else 0.0
        val_pixel_iou = float(np.mean(val_pixel_iou_list)) if val_pixel_iou_list else 0.0
        val_lineiu = float(np.mean(val_lineiu_list)) if val_lineiu_list else 0.0
        val_dr = float(np.mean(val_dr_list)) if val_dr_list else 0.0
        val_ra = float(np.mean(val_ra_list)) if val_ra_list else 0.0
        val_fm = float(np.mean(val_fm_list)) if val_fm_list else 0.0
        alpha_mean = float(np.mean(alpha_means)) if alpha_means else 0.0

        scheduler.step()

        print(
            f"[{run_name}] Epoch {epoch:03d} | "
            f"PixelIoU={val_pixel_iou:.4f} | LineIU={val_lineiu:.4f} | "
            f"DR={val_dr:.4f} | RA={val_ra:.4f} | FM={val_fm:.4f} | "
            f"alpha_mean={alpha_mean:.3f}"
        )

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            "val_pixel_iou": val_pixel_iou,
            "val_lineiu": val_lineiu,
            "val_dr": val_dr,
            "val_ra": val_ra,
            "val_fm": val_fm,
            "alpha_mean": alpha_mean,
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "beta_gate": beta_gate,
        }

        write_header = not os.path.isfile(log_csv)
        with open(log_csv, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
            if write_header:
                writer.writeheader()
            writer.writerow(row)

        state = {
            "model": model.state_dict(),
            "epoch": epoch,
            "val_pixel_iou": val_pixel_iou,
            "val_lineiu": val_lineiu,
            "val_dr": val_dr,
            "val_ra": val_ra,
            "val_fm": val_fm,
            "arch": "unet_textlines_sensitivity",
            "base": BASE,
            "in_ch": in_ch,
            "patch": PATCH,
            "val_stride": VAL_STRIDE,
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "beta_gate": beta_gate,
        }

        torch.save(state, ckpt_last)

        if val_lineiu > best_lineiu:
            best_lineiu = val_lineiu
            best_row = row
            torch.save(state, ckpt_best)
            print(f"Saved best checkpoint: {ckpt_best} | LineIU={val_lineiu:.4f}")

    # -------------------------
    # Test Evaluation
    # -------------------------
    print(f"\n--- Evaluating best model on TEST split for {run_name} ---")
    state = torch.load(ckpt_best, map_location=device)
    model.load_state_dict(state["model"], strict=True)
    model.eval()

    test_ce = 0.0
    test_dice_list = []
    test_pixel_iou_list = []
    test_lineiu_list = []
    test_dr_list = []
    test_ra_list = []
    test_fm_list = []

    test_iter = test_loader

    with torch.no_grad():
        for i, batch in enumerate(test_iter):
            x_full = batch["x"][0].to(device).float()
            y_full = batch["y"][0].to(device).float()

            c, h, w = x_full.shape
            acc = torch.zeros((1, h, w), device=device, dtype=torch.float32)
            wgt = torch.zeros((1, h, w), device=device, dtype=torch.float32)

            ys = list(range(0, max(1, h - PATCH + 1), VAL_STRIDE))
            xs = list(range(0, max(1, w - PATCH + 1), VAL_STRIDE))

            edge_y = max(0, h - PATCH)
            edge_x = max(0, w - PATCH)

            if ys[-1] != edge_y: ys.append(edge_y)
            if xs[-1] != edge_x: xs.append(edge_x)

            win_cache = {}
            for y0 in ys:
                for x0 in xs:
                    y1 = min(h, y0 + PATCH)
                    x1 = min(w, x0 + PATCH)

                    tile = x_full[:, y0:y1, x0:x1].unsqueeze(0).to(device)
                    fused_logits, _ = model(tile)
                    prob = torch.softmax(fused_logits, dim=1)[:, 1:2]

                    th, tw = y1 - y0, x1 - x0
                    key = (th, tw)
                    if key not in win_cache:
                        win_cache[key] = torch.from_numpy(cosine_window_2d(th, tw)).to(device=device, dtype=torch.float32)
                    win = win_cache[key]

                    acc[:, y0:y1, x0:x1] += prob.squeeze(0) * win
                    wgt[:, y0:y1, x0:x1] += win

            prob_full = torch.where(wgt > 0, acc / wgt, acc).clamp(0.0, 1.0)
            
            p = prob_full.clamp(1e-6, 1.0 - 1e-6)
            l_fg = torch.log(p) - torch.log(1.0 - p)
            l_bg = -l_fg
            logits_full = torch.cat([l_bg, l_fg], dim=0).unsqueeze(0)

            target_full = (y_full > 0.5).long().squeeze(0).unsqueeze(0)
            loss_test = ce_loss(logits_full, target_full)
            test_ce += float(loss_test.item())

            pred = (prob_full >= 0.5).float()
            gt = (y_full > 0.5).float()

            tp = (pred * gt).sum().item()
            fp = (pred * (1.0 - gt)).sum().item()
            fn = ((1.0 - pred) * gt).sum().item()

            dice = (2.0 * tp) / (2.0 * tp + fp + fn + 1e-6)
            test_dice_list.append(dice)

            gt_u8 = (gt[0].cpu().numpy().astype(np.uint8)) * 255
            pr_u8 = (pred[0].cpu().numpy().astype(np.uint8)) * 255

            pixel_iou, line_iu, dr, ra, fm = evaluate_metrics_np(gt_u8, pr_u8, thresh=0.75)

            test_pixel_iou_list.append(pixel_iou)
            test_lineiu_list.append(line_iu)
            test_dr_list.append(dr)
            test_ra_list.append(ra)
            test_fm_list.append(fm)

    test_ce /= max(1, len(ds_test))
    test_dice = float(np.mean(test_dice_list)) if test_dice_list else 0.0
    test_pixel_iou = float(np.mean(test_pixel_iou_list)) if test_pixel_iou_list else 0.0
    test_lineiu = float(np.mean(test_lineiu_list)) if test_lineiu_list else 0.0
    test_dr = float(np.mean(test_dr_list)) if test_dr_list else 0.0
    test_ra = float(np.mean(test_ra_list)) if test_ra_list else 0.0
    test_fm = float(np.mean(test_fm_list)) if test_fm_list else 0.0

    print(
        f"TEST RESULTS | "
        f"PixelIoU={test_pixel_iou:.4f} | LineIU={test_lineiu:.4f} | "
        f"DR={test_dr:.4f} | RA={test_ra:.4f} | FM={test_fm:.4f}"
    )

    if best_row is not None:
        best_row["test_pixel_iou"] = test_pixel_iou
        best_row["test_lineiu"] = test_lineiu
        best_row["test_dr"] = test_dr
        best_row["test_ra"] = test_ra
        best_row["test_fm"] = test_fm

    return best_row


# ============================================================
# RUN ALL SENSITIVITY TESTS
# ============================================================

def run_all_sensitivity_tests():
    os.makedirs(OUT_DIR, exist_ok=True)

    summary_path = os.path.join(OUT_DIR, "sensitivity_summary.csv")
    summary_rows = []

    for cfg in RUNS:
        print("\n" + "=" * 90)
        print(f"Starting run: {cfg['name']}")
        print("=" * 90)

        best_row = train_one_run(cfg)

        if best_row is None:
            best_row = {
                "epoch": None,
                "val_pixel_iou": None,
                "val_lineiu": None,
                "val_dr": None,
                "val_ra": None,
                "val_fm": None,
                "alpha_mean": None,
                "test_pixel_iou": None,
                "test_lineiu": None,
                "test_dr": None,
                "test_ra": None,
                "test_fm": None,
            }

        summary_rows.append({
            "run_name": cfg["name"],
            "alpha_min": cfg["alpha_min"],
            "alpha_max": cfg["alpha_max"],
            "beta_gate": cfg["beta_gate"],
            "best_epoch": best_row["epoch"],
            "best_val_pixel_iou": best_row["val_pixel_iou"],
            "best_val_lineiu": best_row["val_lineiu"],
            "best_val_dr": best_row.get("val_dr"),
            "best_val_ra": best_row.get("val_ra"),
            "best_val_fm": best_row["val_fm"],
            "test_pixel_iou": best_row.get("test_pixel_iou"),
            "test_lineiu": best_row.get("test_lineiu"),
            "test_dr": best_row.get("test_dr"),
            "test_ra": best_row.get("test_ra"),
            "test_fm": best_row.get("test_fm"),
        })

    with open(summary_path, "w", newline="") as f:
        fieldnames = list(summary_rows[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)

    print("\nSensitivity summary saved to:")
    print(summary_path)

    print("\nSummary:")
    for row in summary_rows:
        print(row)


if __name__ == "__main__":
    run_all_sensitivity_tests()


Starting run: beta_0.0

Run: beta_0.0
Device: cuda
alpha_min=0.15, alpha_max=1.5, beta_gate=0.0
Loaded Stage 1 model from G:\Best UDIDS\best_maskprob_gauss.pt for dynamic prior generation
[dataset] 3 pairs loaded from G:\historical document analysis\Latin2\images\train
[dataset] 10 pairs loaded from G:\historical document analysis\Latin2\images\val
[dataset] 15 pairs loaded from G:\historical document analysis\Latin2\images\test
[beta_0.0] Epoch 001 | PixelIoU=0.5532 | LineIU=0.1784 | DR=0.0017 | RA=0.0021 | FM=0.0019 | alpha_mean=0.821
Saved best checkpoint: sensitivity_output\beta_0.0\best_lineiu.pt | LineIU=0.1784
[beta_0.0] Epoch 002 | PixelIoU=0.5637 | LineIU=0.2417 | DR=0.0026 | RA=0.0030 | FM=0.0028 | alpha_mean=0.816
Saved best checkpoint: sensitivity_output\beta_0.0\best_lineiu.pt | LineIU=0.2417
[beta_0.0] Epoch 003 | PixelIoU=0.6178 | LineIU=0.6976 | DR=0.1130 | RA=0.0962 | FM=0.1035 | alpha_mean=0.810
Saved best checkpoint: sensitivity_output\beta_0.0\best_lineiu.pt | Line